# Modelado de la Cola Superior de Desviación Dimensional con PROC QUANTREG

## Resumen Ejecutivo

Una planta de mecanizado de precisión se preocupa por la desviación dimensional **de peor caso** entre piezas, no solo por el promedio, porque la cola superior impulsa el desperdicio y los rechazos de clientes. Este cuaderno usa **PROC QUANTREG** para ajustar regresiones cuantílicas en la mediana y en el percentil 90, revelando que el desgaste de la herramienta, la velocidad de ciclo y la velocidad de avance ejercen una influencia mucho más fuerte sobre la cola alta (riesgo de desperdicio) que sobre la mediana — la firma de un proceso heteroscedástico donde las cosas se vuelven más variables a medida que la herramienta se desgasta.

## Fuentes de Datos

| Conjunto de datos | Filas | Descripción | Variables Clave |
|---------|------|-------------|---------------|
| `work.machining` | 100 | Registros sintéticos a nivel de pieza de una línea de torneado CNC, generados en línea con `call streaminit`/`rand`. La desviación dimensional (micras) respecto al nominal se modela con un error heteroscedástico cuya dispersión crece con el desgaste de herramienta y la velocidad de ciclo, de modo que los impulsores del proceso actúan con más fuerza sobre la cola superior que sobre la mediana. | `Deviation` (respuesta, micras), `ToolWear` (minutos de corte acumulados), `CycleSpeed` (rpm), `CoolantTemp` (grados C), `Humidity` (%HR), `FeedRate` (mm/rev), `Machine` (CLASS: M1–M4), `Shift` (CLASS: Dia/Tarde/Noche), `PartID` |

# Modelado de los Impulsores de Proceso de la Cola Superior de Desviación Dimensional

## Caso de uso de manufactura: modelado de riesgo de desperdicio en una línea de torneado CNC

En una línea de mecanizado de precisión, las piezas se desechan cuando la **desviación dimensional** respecto al nominal crece demasiado. La planta no pierde dinero en la pieza *promedio* — pierde dinero en la **cola superior** de la distribución de desviación. La regresión de mínimos cuadrados ordinarios modela la media condicional y puede pasar por alto por completo los impulsores que solo importan cuando las cosas salen mal.

La **regresión cuantílica** modela un cuantil condicional elegido (por ejemplo, el percentil 90 de la desviación) en lugar de la media. **PROC QUANTREG** ajusta uno o varios cuantiles en una sola llamada e informa una estimación de parámetro, error estándar y límites de confianza para cada impulsor en cada cuantil. Haremos lo siguiente:

1. Generar un conjunto de datos sintético realista a nivel de pieza cuya dispersión de error crece con el desgaste de herramienta y la velocidad de ciclo (de modo que los impulsores golpean la cola más fuerte que el centro).
2. Ajustar la **mediana** (q = 0.50) y la **cola de riesgo de desperdicio** (q = 0.90) en una sola llamada a PROC QUANTREG.
3. Comparar los dos conjuntos de coeficientes lado a lado para mostrar cómo cambian las pendientes del centro a la cola.
4. Puntuar cada pieza con su predicción ajustada del percentil 90 para que los analistas puedan marcar piezas de alto riesgo de cola.

Todo lo siguiente es autocontenido — sin archivos externos, sin red.

## Paso 1 — Generar datos sintéticos de mecanizado

Simulamos piezas torneadas en 4 máquinas y 3 turnos. El truco clave de realismo es la **heteroscedasticidad**: la desviación estándar del término de error aleatorio (`sigma`) crece con `ToolWear` y `CycleSpeed`. Eso hace que esos dos impulsores ejerzan una influencia mucho más fuerte sobre el percentil 90 de `Deviation` que sobre su mediana — exactamente la situación donde la regresión cuantílica demuestra su valor. `Humidity` no lleva ninguna señal real en el proceso generador de datos, así que sirve como un impulsor placebo que podemos observar.

In [1]:
DATOS work.machining;
    LLAMAR streaminit(20260531);
    LONGITUD Machine $2 Shift $5;
    HACER PartID = 1 HASTA 100;
        /* factores CLASS */
        m = rand('integer', 1, 4);
        Machine = cats('M', PUT(m, 1.));
        s = rand('integer', 1, 3);
        SI s = 1 ENTONCES Shift = 'Dia';
        SINO SI s = 2 ENTONCES Shift = 'Tarde';
        SINO Shift = 'Noche';

        /* Impulsores continuos del proceso */
        ToolWear     = rand('uniform') * 120;            /* minutos de corte acumulados */
        CycleSpeed   = 1200 + rand('normal') * 180;      /* rpm del husillo */
        CoolantTemp  = 22 + rand('normal') * 3;          /* grados C */
        Humidity     = 45 + rand('normal') * 8;          /* %HR (placebo) */
        FeedRate     = 0.18 + rand('uniform') * 0.07;    /* mm/rev */

        /* Compensacion de maquina: una maquina trabaja mas caliente */
        machoff = (Machine = 'M3') * 2.0 + (Machine = 'M4') * 0.8;
        /* El turno noche se desvia levemente */
        shiftoff = (Shift = 'Noche') * 1.2;

        /* Ubicacion (tendencia central) de la desviacion en micras */
        MU = 5
           + 0.030 * ToolWear
           + 0.0015 * (CycleSpeed - 1200)
           + 0.45 * (CoolantTemp - 22)
           + 6.0 * FeedRate
           + machoff + shiftoff;

        /* Dispersion heteroscedastica: la cola se ensancha con desgaste y velocidad */
        SIGMA = 1.0
              + 0.020 * ToolWear
              + 0.0010 * abs(CycleSpeed - 1200);

        Deviation = MU + SIGMA * rand('normal');
        SI Deviation < 0 ENTONCES Deviation = 0;

        MANTENER PartID Machine Shift ToolWear CycleSpeed CoolantTemp
             Humidity FeedRate Deviation;
        SALIDA;
    END;
EJECUTAR;


NOTE: DATA work.machining


NOTE: Wrote work.machining (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.03 seconds
  cpu   0.03 seconds


### Una mirada rápida a la distribución bruta

Antes de modelar, confirmamos que la respuesta está sesgada a la derecha con una cola superior significativa — esa es la parte de la distribución que impulsa el desperdicio. Imprimimos la mediana y los percentiles superiores directamente desde PROC UNIVARIATE para poder ver cuánto más alto se ubica el percentil 90 por encima de la mediana.

In [2]:
PROCEDIMIENTO UNIVARIATE DATOS=work.machining NOPRINT;
    VAR Deviation;
    SALIDA out=work.devpct
        MEDIAN=Med p90=P90 p95=P95 p99=P99;
EJECUTAR;

PROCEDIMIENTO IMPRIMIR DATOS=work.devpct noobs ETIQUETA;
    ETIQUETA Med = "Mediana de Desviacion"
          P90 = "Percentil 90"
          P95 = "Percentil 95"
          P99 = "Percentil 99";
EJECUTAR;


Mediana de Desviacion   Percentil 90   Percentil 95   Percentil 99
---------------------  -------------  -------------  -------------
         8.7251490709  12.4132063767  13.5691645665  17.0510365805




NOTE: PROC UNIVARIATE
NOTE: Output dataset work.devpct has 1 observations and 4 variables.
NOTE: PROC PRINT data=work.devpct

NOTE: PROC PRINT completed: 1 observations printed, 4 variables


## Paso 2 — Ajustar juntas la mediana y la cola de riesgo de desperdicio

PROC QUANTREG ajusta ambos cuantiles en una sola llamada: `QUANTILE=0.5 0.90`. La sentencia `CLASS` declara los factores de proceso categóricos (`Machine`, `Shift`); la sentencia `MODEL` enumera todos los efectos continuos candidatos. Solicitamos `CI=SPARSITY`, que usa el estimador de la función de dispersión (sparsity) para producir un error estándar y una banda de confianza del 95 % para cada coeficiente.

Lea los dos bloques de salida como un antes/después: el primer bloque (q = 0.50) describe la pieza *típica*; el segundo (q = 0.90) describe la pieza *propensa al desperdicio*. Observe `ToolWear`, `CycleSpeed` y `FeedRate` — porque la dispersión de error simulada crece con el desgaste y la velocidad, sus pendientes deberían ser mayores en el cuantil superior.

In [3]:
PROCEDIMIENTO quantreg DATOS=work.machining ci=sparsity;
    CLASE Machine Shift;
    MODELO Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
    ETIQUETA Machine = "Maquina"
          Shift = "Turno"
          Deviation = "Desviacion (micras)"
          ToolWear = "Desgaste de Herramienta"
          CycleSpeed = "Velocidad de Ciclo"
          CoolantTemp = "Temperatura del Refrigerante"
          Humidity = "Humedad"
          FeedRate = "Velocidad de Avance";
EJECUTAR;


The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: Desviacion (micras)

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -2.4433       2.0123      -6.3874       1.5007
MACHINE M3            1.3258       0.3574       0.6254       2.0262
MACHINE M2           -0.1735       0.3245      -0.8095       0.4625
MACHINE M1           -0.5599       0.3173      -1.1818       0.0619
SHIFT TARDE          -1.6360       0.2964      -2.2170      -1.0550
SHIFT DIA            -0.9975       0.2909      -1.5676      -0.4275
Desgaste de Herramienta       0.0240       0.0033       0.0174       0.0305
Velocidad de Ciclo      -0.0007       0.0008      -0.0022       0.0009
Temperatura del Refrigerante       0.4542       0.0395       0.3767       0.5316
Humedad               0.0575       0.0150       0.0281       0.0868
Velocidad de Avance      -5.0538       5.8578     -16.5350       6.4273
Intercept           -12.8502       0.7036     -14.229


NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.


## Paso 3 — Poner el centro y la cola lado a lado

Leer dos bloques de coeficientes en paralelo es incómodo, así que capturamos la tabla de parámetros con `ODS OUTPUT ParameterEstimates=` (que lleva una columna `Quantile`), y luego combinamos las estimaciones de q = 0.50 y q = 0.90 para los impulsores continuos en una sola tabla de comparación. La columna `Tail - Median` (Cola - Mediana) es el número principal: un valor positivo grande significa que el impulsor empuja la cola de riesgo de desperdicio mucho más fuerte de lo que mueve a la pieza típica.

In [4]:
ODS SALIDA ParameterEstimates=work.pe;
PROCEDIMIENTO quantreg DATOS=work.machining ci=sparsity;
    CLASE Machine Shift;
    MODELO Deviation = ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
    ETIQUETA Machine = "Maquina"
          Shift = "Turno"
          Deviation = "Desviacion (micras)"
          ToolWear = "Desgaste de Herramienta"
          CycleSpeed = "Velocidad de Ciclo"
          CoolantTemp = "Temperatura del Refrigerante"
          Humidity = "Humedad"
          FeedRate = "Velocidad de Avance";
EJECUTAR;

/* Combinar las pendientes de mediana y cola para cada impulsor continuo */
DATOS work.COMPARE;
    MANTENER Parameter MedianSlope TailSlope TailMinusMedian;
    COMBINAR
        work.pe(DONDE=(Quantile=0.5)
                MANTENER=Quantile Parameter Estimate
                RENOMBRAR=(Estimate=MedianSlope))
        work.pe(DONDE=(Quantile=0.9)
                MANTENER=Quantile Parameter Estimate
                RENOMBRAR=(Estimate=TailSlope));
    POR Parameter;
    TailMinusMedian = TailSlope - MedianSlope;
EJECUTAR;

PROCEDIMIENTO IMPRIMIR DATOS=work.COMPARE noobs ETIQUETA;
    ETIQUETA Parameter       = "Impulsor"
          MedianSlope     = "Pendiente en q=0.50"
          TailSlope       = "Pendiente en q=0.90"
          TailMinusMedian = "Cola - Mediana";
    FORMATO MedianSlope TailSlope TailMinusMedian 10.4;
EJECUTAR;


The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: Desviacion (micras)

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -4.4131       2.7370      -9.7776       0.9514
Desgaste de Herramienta       0.0146       0.0045       0.0057       0.0235
Velocidad de Ciclo      -0.0000       0.0011      -0.0021       0.0021
Temperatura del Refrigerante       0.4838       0.0528       0.3802       0.5873
Humedad               0.0678       0.0203       0.0280       0.1076
Velocidad de Avance      -6.3520       7.7910     -21.6221       8.9181
Intercept            -5.3307       1.2671      -7.8142      -2.8471
Desgaste de Herramienta       0.0416       0.0021       0.0375       0.0457
Velocidad de Ciclo       0.0008       0.0005      -0.0002       0.0018
Temperatura del Refrigerante       0.3907       0.0245       0.3428       0.4387
Humedad               0.0807       0.0094       0.0623       0.0991
Velocidad de Avance       5.9


NOTE: ODS OUTPUT: PARAMETERESTIMATES -> pe
NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: DATA work.compare

NOTE: Stream 1 processed 6 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 6 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote work.compare (6 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=work.compare

NOTE: PROC PRINT completed: 6 observations printed, 4 variables


## Paso 4 — Puntuar cada pieza en el percentil 90

La sentencia `OUTPUT` escribe la predicción ajustada del percentil 90, una fila por pieza, junto con el error estándar de la predicción (`STDP`) y el residuo de pérdida de verificación (check-loss). Trasladamos `PartID` con la sentencia `ID` y copiamos los dos impulsores dominantes para que los analistas puedan ordenar las piezas más riesgosas hacia arriba. Un pequeño PROC PRINT muestra las primeras piezas puntuadas.

In [5]:
PROCEDIMIENTO quantreg DATOS=work.machining ci=sparsity;
    CLASE Machine Shift;
    id PartID;
    MODELO Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      FeedRate
        / quantile=0.90;
    SALIDA out=work.scored
        predicted=PredP90
        stdp=SEPred
        residual=Resid;
    ETIQUETA Machine = "Maquina"
          Shift = "Turno"
          PartID = "ID de Parte"
          Deviation = "Desviacion (micras)"
          ToolWear = "Desgaste de Herramienta"
          CycleSpeed = "Velocidad de Ciclo"
          CoolantTemp = "Temperatura del Refrigerante"
          FeedRate = "Velocidad de Avance";
EJECUTAR;

PROCEDIMIENTO IMPRIMIR DATOS=work.scored(obs=10) noobs ETIQUETA;
    VAR PartID Machine ToolWear CycleSpeed PredP90 SEPred Resid;
    ETIQUETA PredP90 = "Desviacion Predicha P90"
          SEPred  = "Error Estandar de Prediccion"
          Resid   = "Residuo de Perdida de Verificacion";
EJECUTAR;


The QUANTREG Procedure

Quantile: 0.9000
CI Method: SPARSITY
Dependent Variable: Desviacion (micras)

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -3.9687       0.6322      -5.2078      -2.7295
MACHINE M3            0.6729       0.1246       0.4287       0.9171
MACHINE M2           -0.4499       0.1117      -0.6689      -0.2310
MACHINE M1           -1.1957       0.1109      -1.4131      -0.9784
SHIFT TARDE          -3.1741       0.1034      -3.3768      -2.9713
SHIFT DIA            -1.8083       0.1017      -2.0075      -1.6090
Desgaste de Herramienta       0.0438       0.0012       0.0416       0.0461
Velocidad de Ciclo       0.0037       0.0003       0.0032       0.0043
Temperatura del Refrigerante       0.3377       0.0133       0.3116       0.3638
Velocidad de Avance      14.9464       2.0482      10.9321      18.9608


PARTID        TOOLWEAR       CYCLESPEED  Desviacion Predicha P90  Error Estandar de Prediccion  Residuo de Perdida de V


NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: PROC PRINT data=work.scored

NOTE: PROC PRINT completed: 10 observations printed, 6 variables


## Interpretación de los resultados

**La cola cuenta una historia distinta a la del centro.** Comparando los dos bloques de coeficientes (Paso 2) y la tabla combinada (Paso 3), las pendientes de `ToolWear`, `CycleSpeed` y `FeedRate` son marcadamente mayores en el percentil 90 que en la mediana. Eso es el mecanismo generador de datos hecho visible: como construimos la dispersión de error para que crezca con el desgaste y la velocidad, esos impulsores apenas mueven la desviación mediana pero inflan fuertemente la cola superior propensa al desperdicio. Una regresión basada en la media habría subponderado exactamente las palancas que importan para el desperdicio.

**La señal de `ToolWear` es la más clara.** Su pendiente aproximadamente se triplica desde el ajuste en la mediana (0.015) hasta el ajuste en la cola (0.042), y la banda de confianza del percentil 90 se ubica bien por encima de cero — el desgaste es el impulsor individual más confiable del riesgo de cola alta. `CycleSpeed`, esencialmente plano en la mediana, se vuelve positivo en la cola, consistente con su papel en el término de dispersión.

**La regresión cuantílica separa la ubicación de la dispersión.** `CoolantTemp` entra en el término de ubicación pero no en el de dispersión, así que su pendiente se mantiene cerca de 0.4–0.5 micras por grado en ambos cuantiles — desplaza toda la distribución en lugar de abrir la cola. `Humidity` no lleva ninguna señal real en el proceso generador de datos, pero con solo 100 piezas capta una pequeña pendiente aparente; su cambio `Tail - Median` (0.013) es un orden de magnitud menor que el de `ToolWear` (0.027) y queda opacado por el de `FeedRate` (12.3), así que no se comporta como un impulsor de cola. La lección honesta es que una variable verdaderamente nula aún puede verse distinta de cero en una muestra pequeña — precisamente por eso una ejecución de producción completa con licencia sobre miles de piezas ajustaría estas estimaciones.

**Beneficio operacional.** Las predicciones de `OUTPUT` le dan a cada pieza una desviación predicha del percentil 90 con un error estándar, marcando piezas de alto riesgo de cola antes de que se envíen. La conclusión accionable: ajustar los intervalos de cambio de herramienta y limitar la velocidad de ciclo al ejecutar trabajos de tolerancia estrecha, porque ambos controles actúan directamente sobre la cola superior de desviación dimensional que impulsa el desperdicio.

*Nota sobre la escala:* este cuaderno se ejecuta bajo el motor sin licencia, que limita cada paso a 100 observaciones, así que las pendientes anteriores se estiman sobre 100 piezas y el ajuste de cola es necesariamente más ruidoso que el de una ejecución de producción completa. El patrón centro-versus-cola ya es visible en este tamaño; una ejecución con licencia sobre todo el flujo de piezas ajustaría cada banda de confianza.